## 1. Setup and Imports

In [1]:
import os
import random
import pandas as pd
import sentencepiece as spm
import math
import sacrebleu
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import zipfile
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.11.0+cpu
CUDA available: False


## 2. Configuration and Hyperparameters

In [2]:
# Data paths
TRAIN_SRC = "../dataset/train/train.zh"
TRAIN_TGT = "../dataset/train/train.vi"
TEST_SRC = "../dataset/private_test/private_test.zh"
SAVE_DIR = "../checkpoints"
SPM_ZH_PREFIX = os.path.join(SAVE_DIR, "spm_zh")
SPM_VI_PREFIX = os.path.join(SAVE_DIR, "spm_vi")

# Model hyperparameters
VOCAB_SIZE = 3000
EMB_SIZE = 64
HID_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 10
LR = 0.01
MAX_LEN = 80
SEED = 42

#Transformer parameters
D_MODEL = 256
NHEAD = 8
NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3
DIM_FF = 512
DROPOUT = 0.2

# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Set random seeds for reproducibility
os.makedirs(SAVE_DIR, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)


Using device: cpu


## 3. Data Loading and Preprocessing

In [3]:
def read_lines(path):
    """Read lines from a text file."""
    with open(path, "r", encoding="utf-8") as f:
        return [l.strip() for l in f if l.strip()]

# Load training and test data
train_src = read_lines(TRAIN_SRC)
train_tgt = read_lines(TRAIN_TGT)
test_src = read_lines(TEST_SRC)

print(f"Training samples: {len(train_src)}")
print(f"Test samples: {len(test_src)}")
print(f"\nExample Chinese sentence: {train_src[0]}")
print(f"Example Vietnamese sentence: {train_tgt[0]}")

Training samples: 32061
Test samples: 1781

Example Chinese sentence: 我 会 给 您 拿 一些 。
Example Vietnamese sentence: Tôi sẽ mang cho bạn một_ít .


## 4. SentencePiece Tokenization

Train BPE tokenizers for both Chinese and Vietnamese.

In [4]:
def train_spm(input_file, model_prefix, vocab_size=VOCAB_SIZE):
    """Train a SentencePiece BPE model."""
    args = (
        f"--input={input_file} --model_prefix={model_prefix} --vocab_size={vocab_size} "
        "--model_type=bpe --character_coverage=1.0 "
        "--pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3"
    )
    spm.SentencePieceTrainer.Train(args)
    print(f"Trained SentencePiece model: {model_prefix}.model")

def load_sp(model_path):
    """Load a trained SentencePiece model."""
    sp = spm.SentencePieceProcessor()
    sp.Load(model_path)
    return sp

In [5]:
# Train Chinese tokenizer
tmp_zh = os.path.join(SAVE_DIR, "tmp_zh.txt")
if not os.path.exists(SPM_ZH_PREFIX + ".model"):
    with open(tmp_zh, "w", encoding="utf-8") as f:
        for s in train_src:
            f.write(s + "\n")
    train_spm(tmp_zh, SPM_ZH_PREFIX)

# Train Vietnamese tokenizer
tmp_vi = os.path.join(SAVE_DIR, "tmp_vi.txt")
if not os.path.exists(SPM_VI_PREFIX + ".model"):
    with open(tmp_vi, "w", encoding="utf-8") as f:
        for s in train_tgt:
            f.write(s + "\n")
    train_spm(tmp_vi, SPM_VI_PREFIX)

# Load tokenizers
sp_zh = load_sp(SPM_ZH_PREFIX + ".model")
sp_vi = load_sp(SPM_VI_PREFIX + ".model")

print(f"\nChinese vocab size: {sp_zh.GetPieceSize()}")
print(f"Vietnamese vocab size: {sp_vi.GetPieceSize()}")

# Test tokenization
test_sent = train_src[0]
tokens = sp_zh.EncodeAsIds(test_sent)
print(f"\nExample tokenization:")
print(f"Original: {test_sent}")
print(f"Token IDs: {tokens[:20]}...")


Chinese vocab size: 3000
Vietnamese vocab size: 3000

Example tokenization:
Original: 我 会 给 您 拿 一些 。
Token IDs: [5, 47, 28, 56, 109, 162, 4]...


## 5. Dataset and DataLoader

In [6]:
class TranslationDataset(Dataset):
    """Dataset for Chinese-Vietnamese translation pairs."""
    
    def __init__(self, src, tgt, sp_src, sp_tgt, max_len=MAX_LEN):
        self.src = src
        self.tgt = tgt
        self.sp_src = sp_src
        self.sp_tgt = sp_tgt
        self.max_len = max_len

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        # Add BOS (2) and EOS (3) tokens
        src_ids = [2] + self.sp_src.EncodeAsIds(self.src[idx])[:self.max_len-2] + [3]
        tgt_ids = [2] + self.sp_tgt.EncodeAsIds(self.tgt[idx])[:self.max_len-2] + [3]
        return torch.tensor(src_ids), torch.tensor(tgt_ids)

def collate_fn(batch):
    """Collate function to pad sequences to the same length."""
    srcs, tgts = zip(*batch)
    max_src = max(len(s) for s in srcs)
    max_tgt = max(len(t) for t in tgts)
    
    # Pad with 0 (PAD token)
    src_pad = torch.zeros(len(batch), max_src, dtype=torch.long)
    tgt_pad = torch.zeros(len(batch), max_tgt, dtype=torch.long)
    
    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_pad[i, :len(s)] = s
        tgt_pad[i, :len(t)] = t
    
    return src_pad, tgt_pad

In [7]:
# Create training dataset and dataloader
# Split data: 90% train, 10% validation
total_samples = len(train_src)
train_size = int(0.9 * total_samples)

train_src_split = train_src[:train_size]
train_tgt_split = train_tgt[:train_size]
valid_src = train_src[train_size:]
valid_tgt = train_tgt[train_size:]

print(f"Total samples: {total_samples}")
print(f"Training samples: {len(train_src_split)}")
print(f"Validation samples: {len(valid_src)}")

# fix num_workers=0 for cpu, if gpu/cuda --> num_workers > 0
dataset = TranslationDataset(train_src_split, train_tgt_split, sp_zh, sp_vi)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=False)

valid_dataset = TranslationDataset(valid_src, valid_tgt, sp_zh, sp_vi)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=False)

print(f"Training batches: {len(dataloader)}")
print(f"Validation batches: {len(valid_loader)}")

Total samples: 32061
Training samples: 28854
Validation samples: 3207
Training batches: 451
Validation batches: 101


## 6. Model Architecture

### Encoder-Decoder with Transformer

In [20]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        pe = torch.zeros(max_len, d_model)
        
        position = torch.arange(
            0, max_len, dtype=torch.float
        ).unsqueeze(1)
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]
    

class TransformerSeq2Seq(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model = 256,
        nhead=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_ff=512,
        dropout=0.1,
        pad_idx=0
    ):
        
        super().__init__()
        
        self.d_model = d_model
        self.pad_idx = pad_idx
        
        self.src_embedding = nn.Embedding(
            src_vocab_size,
            d_model,
            padding_idx=pad_idx
        )
        
        self.tgt_embedding = nn.Embedding(
            tgt_vocab_size,
            d_model,
            padding_idx=pad_idx
        )
        
        self.positional_encoding = PositionalEncoding(d_model)
        
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True
        )
        
        self.fc_out = nn.Linear(
            d_model,
            tgt_vocab_size
        )
        
    def forward(self, src, tgt):

        src_padding_mask = (src == self.pad_idx)
        tgt_padding_mask = (tgt == self.pad_idx)

        tgt_len = tgt.size(1)

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt_len,
            device=tgt.device            )

        src_emb = self.src_embedding(src) * math.sqrt(self.d_model)
        tgt_emb = self.tgt_embedding(tgt) * math.sqrt(self.d_model)

        src_emb = self.positional_encoding(src_emb)
        tgt_emb = self.positional_encoding(tgt_emb)
        output = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
                tgt_mask=tgt_mask,
                src_key_padding_mask=src_padding_mask,
                tgt_key_padding_mask=tgt_padding_mask,
                memory_key_padding_mask=src_padding_mask
            )
        return self.fc_out(output)
            

In [21]:
# Initialize model
# Initialize Transformer model
model = TransformerSeq2Seq(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=256,
    nhead=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_ff=512,
    dropout=0.1,
    pad_idx = 0
).to(DEVICE)

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model has {count_parameters(model):,} trainable parameters")
print(f"\nModel architecture:")
print(model)

Model has 6,261,688 trainable parameters

Model architecture:
TransformerSeq2Seq(
  (src_embedding): Embedding(3000, 256, padding_idx=0)
  (tgt_embedding): Embedding(3000, 256, padding_idx=0)
  (positional_encoding): PositionalEncoding()
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
 

## 7. Training Functions

In [ ]:
def train_epoch(
    model,
    dataloader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0

    for src, tgt in tqdm(dataloader):

        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        optimizer.zero_grad()

        logits = model(
            src,
            tgt_input
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_output.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate_bleu(model, dataloader, sp_tgt):
    model.eval()

    hypotheses = []
    references = []

    BOS_ID = sp_tgt.bos_id()
    EOS_ID = sp_tgt.eos_id()
    PAD_ID = sp_tgt.pad_id()

    for src, tgt in tqdm(dataloader, desc="Evaluating BLEU"):
        src = src.to(DEVICE)

        batch_size = src.size(0)

        # Decoder bắt đầu bằng <BOS>
        generated = torch.full(
            (batch_size, 1),
            BOS_ID,
            dtype=torch.long,
            device=DEVICE
        )

        finished = torch.zeros(
            batch_size,
            dtype=torch.bool,
            device=DEVICE
        )

        # autoregressive decoding
        for _ in range(MAX_LEN):
            logits = model(src, generated)

            # chỉ lấy prediction của token cuối
            next_token = logits[:, -1, :].argmax(dim=-1)

            generated = torch.cat(
                [generated, next_token.unsqueeze(1)],
                dim=1
            )

            finished |= (next_token == EOS_ID)

            if finished.all():
                break

        generated = generated.cpu().tolist()
        tgt = tgt.tolist()

        for pred_ids, ref_ids in zip(generated, tgt):

            # bỏ BOS
            pred_ids = pred_ids[1:]

            # cắt tại EOS
            if EOS_ID in pred_ids:
                pred_ids = pred_ids[:pred_ids.index(EOS_ID)]

            # bỏ special tokens/padding
            pred_ids = [
                x for x in pred_ids
                if x not in (PAD_ID, BOS_ID, EOS_ID)
            ]

            ref_ids = [
                x for x in ref_ids
                if x not in (PAD_ID, BOS_ID, EOS_ID)
            ]

            pred_text = sp_tgt.decode(pred_ids)
            ref_text = sp_tgt.decode(ref_ids)

            hypotheses.append(pred_text)
            references.append(ref_text)

    bleu = sacrebleu.corpus_bleu(
        hypotheses,
        [references]
    )

    return bleu.score

## 8. Training Loop

In [24]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)  # Ignore padding
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.98),
    eps=1e-9,
)

print("Starting training...\n")

# Training loop with best model saving
best_bleu = 0.0
best_model_path = os.path.join(SAVE_DIR, "best_model.pt")

for epoch in range(1, EPOCHS + 1):
    print(f"\n===Epoch {epoch} / {EPOCHS}===")
    loss = train_epoch(model, dataloader, criterion, optimizer, device=DEVICE)
    bleu = evaluate_bleu(model, valid_loader, sp_vi)
    
    
    # Save best model
    if bleu > best_bleu:
        best_bleu = bleu
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'bleu': bleu,
            'loss': loss
        }, best_model_path)
        print(f"Epoch {epoch:02d} | Loss={loss:.3f} | SacreBLEU={bleu:.2f} Best model saved!")
    else:
        print(f"Epoch {epoch:02d} | Loss={loss:.3f} | SacreBLEU={bleu:.2f}")

print(f"\nTraining completed!")
print(f"Best validation BLEU: {best_bleu:.2f}")
print(f"Best model saved to: {best_model_path}")

# Load best model for inference
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']}")

Starting training...


===Epoch 1 / 10===


  0%|          | 0/451 [00:00<?, ?it/s]c:\Users\THINH DINH\miniconda3\Lib\site-packages\torch\nn\modules\activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
  3%|▎         | 13/451 [00:15<08:37,  1.18s/it]


KeyboardInterrupt: 

## 9. Translation on Test Set

In [ ]:
@torch.no_grad()
def translate_sentence(model, sentence, sp_src, sp_tgt):
    model.eval()

    src_ids = (
        [2]
        + sp_src.EncodeAsIds(sentence)[:MAX_LEN - 2]
        + [3]
    )

    src = torch.tensor(
        src_ids,
        dtype=torch.long,
        device=DEVICE
    ).unsqueeze(0)

    # bắt đầu bằng BOS
    generated = torch.tensor(
        [[2]],
        dtype=torch.long,
        device=DEVICE
    )

    for _ in range(MAX_LEN):
        logits = model(src, generated)

        next_token = logits[:, -1, :].argmax(dim=-1)

        token_id = next_token.item()

        if token_id == 3:  # EOS
            break

        generated = torch.cat(
            [generated, next_token.unsqueeze(1)],
            dim=1
        )

    token_ids = generated[0, 1:].tolist()

    return sp_tgt.DecodeIds(token_ids)

In [ ]:
@torch.no_grad()
def translate_test(model, test_src, sp_src, sp_tgt, out_path):
    outputs = []

    for sentence in tqdm(test_src, desc="Translating"):
        translation = translate_sentence(
            model,
            sentence,
            sp_src,
            sp_tgt
        )

        outputs.append(translation)

    df = pd.DataFrame({
        "tieng_trung": test_src,
        "tieng_viet": outputs
    })

    df.to_csv(
        out_path,
        index=False,
        encoding="utf-8-sig"
    )

    return out_path

In [20]:
# Translate test set
submission_path = os.path.join("nlp_submission.csv")
translate_test(model, test_src, sp_zh, sp_vi, submission_path)

print(f"Translation completed!")
print(f"Saved to: {submission_path}")

# Display sample translations
df_result = pd.read_csv(submission_path)
print("\nSample translations:")
print(df_result.head(10))

Translating: 100%|██████████| 1781/1781 [00:09<00:00, 185.22it/s]

Translation completed!
Saved to: nlp_submission.csv

Sample translations:
                              tieng_trung  \
0   就 我 来说 , 通常 都 是 经营 的 问题 ， 很 少 带来 欢乐 。   
1                          你 有 哪些 运动 器材 ？   
2                          酒店 有 会议 设施 吗 ？   
3                        我 会 度 过 这 段 时间 。   
4                        我 需要 一 张 填表 谢谢 。   
5                             我 有 慢性 哮喘 。   
6                         坐 船 过 池塘 要 多久 ？   
7                          她 想 要 一 杯 啤酒 。   
8                                  去 纽约 。   
9  你 今晚 八点半 左右 不 是 偶然 在 十二 G 街 的 拐角 是 吗 ？   

                                         tieng_viet  
0                   Vâng , tôi tôi có một tôi______  
1                                  Bạn có________ ?  
2                        Luốc có___________ không ?  
3                        Tôi sẽ mất thời_gian này .  
4   Làm vuiơn cho tôi một một một một một một một .  
5                           Tôi có một một_____ . .  
6                     Tôi mất bao_lâu mất bao_lâu ? 

## 10. Create Submission ZIP

In [21]:
# Create ZIP file for submission
zip_path = os.path.join("private_submission.zip")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(submission_path, arcname="nlp_submission.csv")

print(f"Submission ZIP created: {zip_path}")
print(f"File size: {os.path.getsize(zip_path) / 1024:.2f} KB")

Submission ZIP created: private_submission.zip
File size: 46.85 KB
